# FIFA World Cup 2026 — Teams Data Collection

This notebook collects the official FIFA World Cup 2026 teams data using the FIFA API.

The collected data will be cleaned, validated, and exported for use in the FIFA World Cup 2026 SQL database.

## 1. Project Setup

Import the Python libraries required to connect to the FIFA API and work with the collected data.

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. FIFA API Connection

We will connect directly to the official FIFA World Cup 2026 standings API.

This endpoint provides the 48 teams participating in the tournament, along with their groups and FIFA identifiers.

In [3]:
# Test the official FIFA World Cup 2026 Teams endpoint

group_standings_url = (
    "https://api.fifa.com/api/v3/calendar/"
    "17/285023/289273/standing"
    "?language=en&count=200"
)

response = requests.get(
    group_standings_url,
    headers=headers,
    timeout=20
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Content length:", len(response.text))
print("Connection successful:", response.status_code == 200)

Status code: 200
Content type: application/json; charset=utf-8
Content length: 109707
Connection successful: True


## 3. Teams Data Extraction

The FIFA API response contains the official World Cup 2026 group standings.

We will request the data and convert the JSON response into a Python dictionary so we can extract the 48 participating teams.

In [4]:
# Request the World Cup 2026 teams data

group_response = requests.get(
    group_standings_url,
    headers=headers,
    timeout=20
)

group_data = group_response.json()

print("Data received successfully.")
print("\nTop-level keys:")
print(group_data.keys())

print("\nNumber of records:")
print(len(group_data["Results"]))

Data received successfully.

Top-level keys:
dict_keys(['ContinuationToken', 'ContinuationHash', 'Results'])

Number of records:
48


## 4. Extract Team Information

From each FIFA API record, we will extract the information required for our `Teams` SQL table:

- FIFA Team ID
- Team Name
- Country Code
- Confederation
- Group Name

The data will be stored in a pandas DataFrame for cleaning and validation.

In [5]:
# Extract the official team information

teams_data = []

for item in group_data["Results"]:
    team = item["Team"]

    team_name = next(
        name["Description"]
        for name in team["Name"]
        if name["Locale"] == "en-GB"
    )

    group_name = next(
        group["Description"]
        for group in item["Group"]
        if group["Locale"] == "en-GB"
    )

    teams_data.append({
        "FIFA_Team_ID": team["IdTeam"],
        "TeamName": team_name,
        "CountryCode": team["IdCountry"],
        "Confederation": team["IdConfederation"],
        "GroupName": group_name
    })

teams_df = pd.DataFrame(teams_data)

print("Teams extracted successfully.")
print("Dataset shape:", teams_df.shape)

display(teams_df)

Teams extracted successfully.
Dataset shape: (48, 5)


,FIFA_Team_ID,TeamName,CountryCode,Confederation,GroupName
0,43922,Argentina,ARG,CONMEBOL,Group J
1,43935,Belgium,BEL,UEFA,Group G
2,43924,Brazil,BRA,CONMEBOL,Group C
3,43926,Colombia,COL,CONMEBOL,Group K
4,43942,England,ENG,UEFA,Group L
5,43946,France,FRA,UEFA,Group I
6,43948,Germany,GER,UEFA,Group E
7,43911,Mexico,MEX,CONCACAF,Group A
8,43960,Netherlands,NED,UEFA,Group F
9,43969,Spain,ESP,UEFA,Group H


## 5. Data Validation

Before adding our own `TeamID` and exporting the dataset, we will validate the official FIFA data.

We will check:

- Exactly 48 teams
- Unique FIFA Team IDs
- Unique team names
- Unique country codes
- Exactly 12 groups
- Exactly 4 teams in every group
- No missing values in the required columns

In [6]:
# Validate the Teams dataset

required_columns = [
    "FIFA_Team_ID",
    "TeamName",
    "CountryCode",
    "Confederation",
    "GroupName"
]

print("=== BASIC VALIDATION ===")
print("Total teams:", len(teams_df))
print("Unique FIFA Team IDs:", teams_df["FIFA_Team_ID"].nunique())
print("Unique team names:", teams_df["TeamName"].nunique())
print("Unique country codes:", teams_df["CountryCode"].nunique())
print("Number of groups:", teams_df["GroupName"].nunique())

print("\n=== TEAMS PER GROUP ===")
group_counts = teams_df["GroupName"].value_counts().sort_index()
print(group_counts)

print("\n=== MISSING VALUES ===")
print(teams_df[required_columns].isnull().sum())

print("\n=== DUPLICATES ===")
print("Duplicate FIFA Team IDs:", teams_df["FIFA_Team_ID"].duplicated().sum())
print("Duplicate team names:", teams_df["TeamName"].duplicated().sum())
print("Duplicate country codes:", teams_df["CountryCode"].duplicated().sum())

print("\n=== FINAL CHECK ===")

valid = (
    len(teams_df) == 48
    and teams_df["FIFA_Team_ID"].nunique() == 48
    and teams_df["TeamName"].nunique() == 48
    and teams_df["CountryCode"].nunique() == 48
    and teams_df["GroupName"].nunique() == 12
    and group_counts.eq(4).all()
    and teams_df[required_columns].isnull().sum().sum() == 0
    and teams_df["FIFA_Team_ID"].duplicated().sum() == 0
    and teams_df["TeamName"].duplicated().sum() == 0
    and teams_df["CountryCode"].duplicated().sum() == 0
)

print("Dataset valid:", valid)

=== BASIC VALIDATION ===
Total teams: 48
Unique FIFA Team IDs: 48
Unique team names: 48
Unique country codes: 48
Number of groups: 12

=== TEAMS PER GROUP ===
GroupName
Group A    4
Group B    4
Group C    4
Group D    4
Group E    4
Group F    4
Group G    4
Group H    4
Group I    4
Group J    4
Group K    4
Group L    4
Name: count, dtype: int64

=== MISSING VALUES ===
FIFA_Team_ID     0
TeamName         0
CountryCode      0
Confederation    0
GroupName        0
dtype: int64

=== DUPLICATES ===
Duplicate FIFA Team IDs: 0
Duplicate team names: 0
Duplicate country codes: 0

=== FINAL CHECK ===
Dataset valid: True


## 6. Create TeamID

The FIFA API provides its own team identifier, but our SQL database uses a separate `TeamID` as the primary key.

We will create a sequential `TeamID` from 1 to 48 while keeping the original FIFA Team ID for reference.

In [7]:
# Create our own TeamID for the SQL database

teams_df.insert(
    0,
    "TeamID",
    range(1, len(teams_df) + 1)
)

print("TeamID created successfully.")
print("TeamID range:", teams_df["TeamID"].min(), "to", teams_df["TeamID"].max())
print("Unique TeamIDs:", teams_df["TeamID"].nunique())

display(teams_df)

TeamID created successfully.
TeamID range: 1 to 48
Unique TeamIDs: 48


,TeamID,FIFA_Team_ID,TeamName,CountryCode,Confederation,GroupName
0,1,43922,Argentina,ARG,CONMEBOL,Group J
1,2,43935,Belgium,BEL,UEFA,Group G
2,3,43924,Brazil,BRA,CONMEBOL,Group C
3,4,43926,Colombia,COL,CONMEBOL,Group K
4,5,43942,England,ENG,UEFA,Group L
5,6,43946,France,FRA,UEFA,Group I
6,7,43948,Germany,GER,UEFA,Group E
7,8,43911,Mexico,MEX,CONCACAF,Group A
8,9,43960,Netherlands,NED,UEFA,Group F
9,10,43969,Spain,ESP,UEFA,Group H


## 7. Prepare the Teams Dataset

The SQL `Teams` table uses our internal `TeamID` as the primary key.

We will:
- Keep the official FIFA Team ID in our collected dataset for reference.
- Create the SQL-ready dataset using the columns defined in our database.
- Add `HostCountry` as a nullable column because only teams representing the host countries need a host-country value.

In [8]:
# Prepare the Teams dataset for SQL

teams_sql_df = teams_df[
    [
        "TeamID",
        "TeamName",
        "CountryCode",
        "Confederation",
        "GroupName"
    ]
].copy()

# Add HostCountry as a nullable column
teams_sql_df["HostCountry"] = pd.NA

print("SQL-ready Teams dataset created.")
print("Shape:", teams_sql_df.shape)

display(teams_sql_df)

SQL-ready Teams dataset created.
Shape: (48, 6)


,TeamID,TeamName,CountryCode,Confederation,GroupName,HostCountry
0,1,Argentina,ARG,CONMEBOL,Group J,<NA>
1,2,Belgium,BEL,UEFA,Group G,<NA>
2,3,Brazil,BRA,CONMEBOL,Group C,<NA>
3,4,Colombia,COL,CONMEBOL,Group K,<NA>
4,5,England,ENG,UEFA,Group L,<NA>
5,6,France,FRA,UEFA,Group I,<NA>
6,7,Germany,GER,UEFA,Group E,<NA>
7,8,Mexico,MEX,CONCACAF,Group A,<NA>
8,9,Netherlands,NED,UEFA,Group F,<NA>
9,10,Spain,ESP,UEFA,Group H,<NA>


### 7.1 Add Host Countries

The 2026 FIFA World Cup is hosted by Canada, Mexico, and the United States.

We will populate `HostCountry` only for the teams representing the three host nations. All other teams will remain NULL because they are not host teams.

In [9]:
# Add host-country information

host_countries = {
    "CAN": "Canada",
    "MEX": "Mexico",
    "USA": "United States"
}

teams_sql_df["HostCountry"] = teams_sql_df["CountryCode"].map(host_countries)

print("Host countries added.")

display(
    teams_sql_df[
        teams_sql_df["HostCountry"].notna()
    ]
)

Host countries added.


,TeamID,TeamName,CountryCode,Confederation,GroupName,HostCountry
7,8,Mexico,MEX,CONCACAF,Group A,Mexico
11,12,USA,USA,CONCACAF,Group D,United States
15,16,Canada,CAN,CONCACAF,Group B,Canada


### 7.2 Final Teams Dataset Check

Before exporting the Teams dataset, we will verify that:

- There are exactly 48 teams.
- TeamID values are unique and range from 1 to 48.
- Exactly 3 teams have a HostCountry value.
- The host teams are Canada, Mexico, and USA.
- The dataset contains the six columns required for the SQL Teams table.

In [10]:
# Final check of the Teams SQL-ready dataset

print("Number of teams:", len(teams_sql_df))
print("Unique TeamIDs:", teams_sql_df["TeamID"].nunique())
print("TeamID range:", teams_sql_df["TeamID"].min(), "to", teams_sql_df["TeamID"].max())

print("\nHost teams:")
print(
    teams_sql_df.loc[
        teams_sql_df["HostCountry"].notna(),
        ["TeamID", "TeamName", "CountryCode", "HostCountry"]
    ].to_string(index=False)
)

print("\nNumber of host teams:", teams_sql_df["HostCountry"].notna().sum())

print("\nColumns:")
print(list(teams_sql_df.columns))

Number of teams: 48
Unique TeamIDs: 48
TeamID range: 1 to 48

Host teams:
 TeamID TeamName CountryCode   HostCountry
      8   Mexico         MEX        Mexico
     12      USA         USA United States
     16   Canada         CAN        Canada

Number of host teams: 3

Columns:
['TeamID', 'TeamName', 'CountryCode', 'Confederation', 'GroupName', 'HostCountry']


## 8. Export Teams Dataset

The validated Teams dataset will be saved inside the project's `Data` folder.

The file path is created dynamically so that the project remains portable and can work on another laptop without changing a hard-coded Windows path.

In [12]:
# Export the Teams dataset to the project's Data folder

from pathlib import Path

# Find the project folder from the notebook location
project_folder = Path.cwd().parent

# Create the Data folder if it does not already exist
data_folder = project_folder / "Data"
data_folder.mkdir(parents=True, exist_ok=True)

# Define the CSV file path
teams_csv_path = data_folder / "Teams.csv"

# Export the dataset
teams_sql_df.to_csv(
    teams_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Teams dataset exported successfully.")
print("File location:")
print(teams_csv_path)

Teams dataset exported successfully.
File location:
d:\ai course hw\FIFA_WorldCup_2026_Project\Data\Teams.csv


### 8.1 Verify Exported CSV

We will verify that the exported `Teams.csv` file exists, can be read successfully, and contains the same 48 validated teams.

In [13]:
# Verify the exported Teams.csv file

print("File exists:", teams_csv_path.exists())
print("File name:", teams_csv_path.name)
print("File size:", teams_csv_path.stat().st_size, "bytes")

# Read the exported CSV
teams_check_df = pd.read_csv(teams_csv_path)

print("\nCSV shape:", teams_check_df.shape)
print("Number of teams:", len(teams_check_df))

print("\nCSV columns:")
print(list(teams_check_df.columns))

print("\nFirst 5 rows:")
display(teams_check_df.head())

File exists: True
File name: Teams.csv
File size: 1593 bytes

CSV shape: (48, 6)
Number of teams: 48

CSV columns:
['TeamID', 'TeamName', 'CountryCode', 'Confederation', 'GroupName', 'HostCountry']

First 5 rows:


,TeamID,TeamName,CountryCode,Confederation,GroupName,HostCountry
0,1,Argentina,ARG,CONMEBOL,Group J,NaN
1,2,Belgium,BEL,UEFA,Group G,NaN
2,3,Brazil,BRA,CONMEBOL,Group C,NaN
3,4,Colombia,COL,CONMEBOL,Group K,NaN
4,5,England,ENG,UEFA,Group L,NaN


## 9. Final Teams CSV Validation

The exported CSV will be validated one final time to ensure that the file we will use in the project contains valid and complete Teams data.

In [14]:
# Final validation of the exported Teams CSV

required_columns = [
    "TeamID",
    "TeamName",
    "CountryCode",
    "Confederation",
    "GroupName",
    "HostCountry"
]

final_valid = (
    len(teams_check_df) == 48
    and teams_check_df[required_columns].isnull().sum().sum() == 45
    and teams_check_df["TeamID"].nunique() == 48
    and teams_check_df["TeamID"].min() == 1
    and teams_check_df["TeamID"].max() == 48
    and teams_check_df["TeamName"].nunique() == 48
    and teams_check_df["CountryCode"].nunique() == 48
    and teams_check_df["GroupName"].nunique() == 12
    and teams_check_df["GroupName"].value_counts().eq(4).all()
    and teams_check_df["HostCountry"].notna().sum() == 3
)

print("=== FINAL CSV VALIDATION ===")
print("Rows:", len(teams_check_df))
print("Unique TeamIDs:", teams_check_df["TeamID"].nunique())
print("Unique teams:", teams_check_df["TeamName"].nunique())
print("Groups:", teams_check_df["GroupName"].nunique())
print("Host teams:", teams_check_df["HostCountry"].notna().sum())

print("\nFinal CSV valid:", final_valid)

=== FINAL CSV VALIDATION ===
Rows: 48
Unique TeamIDs: 48
Unique teams: 48
Groups: 12
Host teams: 3

Final CSV valid: True
